### Transformar Datos de "Employee Training"
1. Extrae la "Fecha" y "Hora" del campo "completition_date" y crea nuevas columnas llamadas "completition_date" y "completition_time".
2. Asignar al campo "status" los siguientes valores decriptivos: <br>
   (1: Not Started, 2: In Progress, 3: Completed)
3. Escribir los datos "transformados" en el schema "silver

In [0]:
df = spark.table("zenviro.bronze.employee_training_py")
display(df)

#### 1. Extrae la "Fecha" y "Hora" del campo "completition_date"

In [0]:
from pyspark.sql.functions import date_format
df_extracted = (
    df
    .select(
        "employee_id",
        "training_id",
        date_format("completition_date", "yyyy-MM-dd").cast("date").alias("completition_date"),
        date_format("completition_date", "hh:mm:ss").alias("completition_time"),
        "status"
    )
)
display(df_extracted)

#### 2. Asignar al campo "status" los siguientes valores decriptivos:
##### (1: Not Started, 2: In Progress, 3: Completed)

In [0]:
from pyspark.sql.functions import when
df_mapped = (
    df_extracted
    .select(
        "employee_id",
        "training_id",
        "completition_date",
        "completition_time",
        when(df_extracted.status == 1, "Not Started")
        .when(df_extracted.status == 2, "In Progress")
        .when(df_extracted.status == 3, "Completed")
        .alias("status")
    )
)
display(df_mapped)

#### 3. Escribir los datos "transformados" en el schema "silver

In [0]:
df_mapped.writeTo("zenviro.silver.employee_training_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.employee_training_py"))